# RAG Evaluation with RAGAS + ChromaDB + Boeing AI Endpoints

**Stack:** ChromaDB · Boeing `text-embedding-3-small` · Boeing `gpt-4o-mini` · RAGAS v0.2+

| Metric | What it measures | Needs ground truth? |
|---|---|---|
| `faithfulness` | Answer is grounded in context (no hallucination) | No |
| `answer_relevancy` | Answer is on-topic for the question | No |
| `context_precision` | Retrieved chunks are useful and well-ranked | Yes |
| `context_recall` | Context fully covers the ground truth | Yes |
| `answer_correctness` | Answer matches ground truth factually | Yes |

This notebook uses **mixed-quality** Q&A pairs — some good, some partial, some intentionally unanswerable — so scores are **realistic and varied** (not all 1.0).

**Changed from OpenAI to Boeing AI (BCAI) endpoints** using `BoeingChatModel` and `BoeingEmbeddings`.

## 1. Install Dependencies

In [ ]:
!pip install \
ragas==0.2.15 \
langchain==0.3.26 \
langchain-community==0.3.27 \
langchain-core==0.3.68

# NOTE: boeing_chat_model and boeing_embeddings must already be installed.
# langchain-openai is NOT needed — we use Boeing endpoints instead.

## 2. Imports and API Key

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

UDAL_PAT = os.getenv('UDAL_PAT')
if not UDAL_PAT:
    raise ValueError('UDAL_PAT not found in .env file')

import chromadb

# Boeing AI imports (same as Simple RAG notebook)
from boeing_chat_model import BoeingChatModel
from boeing_embeddings import BoeingEmbeddings

# RAGAS v0.2+ uses class-based metrics and EvaluationDataset
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    AnswerCorrectness,
)
# RAGAS wrappers for LangChain-compatible LLMs and Embeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print('All imports OK')

# ============================================================
# BOEING AI CLIENTS
# ============================================================

# Chat model for RAGAS judge (temperature=0 for deterministic eval)
boeing_chat = BoeingChatModel(
    udal_pat=UDAL_PAT,
    model='gpt-4o-mini',
    max_tokens=1024,
    temperature=0,
)

# Embedding model for RAGAS
boeing_emb = BoeingEmbeddings(
    udal_pat=UDAL_PAT,
    model='text-embedding-3-small',
)

# Wrap with RAGAS LangChain wrappers
# (BoeingChatModel and BoeingEmbeddings follow the LangChain interface)
ragas_llm = LangchainLLMWrapper(boeing_chat)
ragas_emb = LangchainEmbeddingsWrapper(boeing_emb)

# Instantiate each metric as an object (required in RAGAS v0.2+)
metric_faithfulness       = Faithfulness(llm=ragas_llm)
metric_answer_relevancy   = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)
metric_context_precision  = LLMContextPrecisionWithReference(llm=ragas_llm)
metric_context_recall     = LLMContextRecall(llm=ragas_llm)
metric_answer_correctness = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb)

print('RAGAS metrics instantiated (using Boeing AI endpoints)')

## 3. Knowledge Base

A small corpus about RAG / ML topics.  
**Important:** some questions asked later are NOT covered here — that is intentional, so the retriever returns irrelevant context and scores drop realistically.

In [ ]:
documents = [
    # doc_0
    "Retrieval-Augmented Generation (RAG) is an AI framework that enhances large language models "
    "by connecting them to external knowledge bases. Instead of relying solely on parametric memory, "
    "RAG retrieves relevant documents at inference time and conditions the generation on them.",

    # doc_1
    "ChromaDB is an open-source, AI-native vector database. It allows developers to store embeddings "
    "alongside metadata and perform low-latency similarity searches. ChromaDB can run in-memory or "
    "persisted to disk and integrates natively with LangChain and LlamaIndex.",

    # doc_2
    "GPT-4o-mini is a small, cost-efficient multimodal model released by OpenAI in July 2024. "
    "It scores 82% on the MMLU benchmark, supports a 128k context length, and achieves better "
    "performance than GPT-3.5 Turbo while being significantly cheaper than GPT-4o.",

    # doc_3
    "RAGAS (Retrieval Augmented Generation Assessment) is an open-source Python framework for "
    "evaluating RAG pipelines. It measures faithfulness, answer relevancy, context precision, "
    "context recall, and answer correctness using LLM-based judges.",

    # doc_4
    "Faithfulness in RAGAS checks whether every claim in the generated answer can be inferred "
    "from the retrieved context. It decomposes the answer into atomic claims and verifies each "
    "one. A score of 1.0 means zero hallucination.",

    # doc_5
    "Context recall measures what fraction of the ground-truth answer's key claims are covered "
    "by the retrieved context. A low context recall means the retriever failed to fetch documents "
    "containing the information needed to answer the question.",

    # doc_6
    "Vector embeddings are dense numerical representations of text or other data. "
    "They capture semantic meaning so that similar items cluster together in the embedding space "
    "and can be found with similarity search.",

    # doc_7  intentionally off-topic to pollute retrieval for hard questions
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python emphasises code readability and uses significant indentation. It supports procedural, "
    "object-oriented, and functional programming paradigms.",

    # doc_8  partial info only
    "Chunking strategies in RAG include fixed-size chunking, sentence-level chunking, and "
    "semantic chunking. The chunk size significantly affects retrieval quality.",

    # doc_9
    "LangChain is a framework for building LLM-powered applications. It provides abstractions "
    "for chains, agents, retrievers, and memory modules that simplify complex workflows.",
]

doc_ids = [f'doc_{i}' for i in range(len(documents))]
print(f'{len(documents)} documents ready')

## 4. Build ChromaDB Vector Store

In [ ]:
# Custom ChromaDB embedding function using BoeingEmbeddings
class BoeingEmbeddingFunction:
    """Wraps BoeingEmbeddings for use as a ChromaDB embedding function."""
    def __init__(self, boeing_emb):
        self._emb = boeing_emb

    def __call__(self, input):
        return self._emb.embed_documents(input)

boeing_ef = BoeingEmbeddingFunction(boeing_emb)

# EphemeralClient = in-memory, no persistence needed for eval
chroma_client = chromadb.EphemeralClient()

collection = chroma_client.create_collection(
    name='rag_kgb',
    embedding_function=boeing_ef,
    metadata={'hnsw:space': 'cosine'}
)

collection.add(documents=documents, ids=doc_ids)
print(f'ChromaDB collection: {collection.count()} documents')

## 5. RAG Pipeline

In [ ]:
# RAG chat client (separate instance for generation, can use different temp)
rag_chat = BoeingChatModel(
    udal_pat=UDAL_PAT,
    model='gpt-4o-mini',
    max_tokens=500,
    temperature=0.0,
)

def retrieve(query: str, n_results: int = 3):
    """Retrieve top-k chunks from ChromaDB."""
    res = collection.query(query_texts=[query], n_results=n_results)
    return res['documents'][0]   # list[str]


def generate(query: str, context_chunks: list, temperature: float = 0.0):
    """Generate an answer with Boeing GPT, grounded in context."""
    context = '\n\n'.join(context_chunks)
    system  = (
        'You are a helpful assistant. '
        'Answer using ONLY the context provided. '
        'If the context does not contain enough information, '
        'say exactly: "The context does not contain this information."'
    )
    user = f'Context:\n{context}\n\nQuestion: {query}\nAnswer:'
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user},
    ]
    response = rag_chat.invoke(messages)
    return response.content.strip()


def rag(query: str, n_results: int = 3):
    """Full RAG pipeline."""
    contexts = retrieve(query, n_results)
    answer   = generate(query, contexts)
    return answer, contexts


# Quick smoke test
q = 'What is ChromaDB?'
a, ctx = rag(q)
print(f'Q: {q}\nA: {a}\n')
for i, c in enumerate(ctx, 1):
    print(f'  [{i}] {c[:90]}...')

## 6. Mixed-Quality Evaluation Dataset

Three tiers:
- **good** — answer is fully in the KB → expect high scores across all metrics  
- **medium** — answer is only partially in the KB → middle scores  
- **bad** — answer is NOT in the KB at all → the model should say it doesn't know, causing low relevancy, recall, and correctness

In [ ]:
eval_set = [
    # ---- GOOD: fully covered by KB ----------------------------------------
    {
        'question':     'What is ChromaDB and what are its main features?',
        'ground_truth': 'ChromaDB is an open-source AI-native vector database that stores embeddings '
                        'alongside metadata and performs low-latency similarity search. It can run '
                        'in-memory or persisted to disk and integrates with LangChain and LlamaIndex.',
        'quality':      'good',
    },
    {
        'question':     'What does faithfulness measure in RAGAS and what does a score of 1.0 mean?',
        'ground_truth': 'Faithfulness checks whether every claim in the generated answer can be '
                        'inferred from the retrieved context by decomposing the answer into atomic '
                        'claims and verifying each one. A score of 1.0 means zero hallucination.',
        'quality':      'good',
    },

    # ---- MEDIUM: partially covered -----------------------------------------
    {
        'question':     'What chunking strategies exist in RAG and how do they affect quality?',
        'ground_truth': 'Common chunking strategies include fixed-size, sentence-level, and semantic '
                        'chunking. Chunk size significantly affects retrieval quality. Smaller chunks '
                        'improve precision while larger chunks improve recall but may add noise. '
                        'Overlap between chunks helps preserve continuity across chunk boundaries.',
        'quality':      'medium',  # KB only has 1 partial sentence on this
    },
    {
        'question':     'How does GPT-4o-mini compare with GPT-4o in terms of cost and performance?',
        'ground_truth': 'GPT-4o-mini is significantly cheaper than GPT-4o and scores 82% on MMLU. '
                        'It outperforms GPT-3.5 Turbo. However GPT-4o-mini lacks some advanced '
                        'reasoning capabilities that GPT-4o has, and GPT-4o supports longer context '
                        'windows and better multilingual performance.',
        'quality':      'medium',  # KB has partial comparison info
    },

    # ---- BAD: answer NOT in KB at all → retriever returns irrelevant chunks -
    {
        'question':     'What is the price per million tokens for GPT-4o-mini on the OpenAI API?',
        'ground_truth': 'GPT-4o-mini costs $0.15 per million input tokens and $0.60 per million '
                        'output tokens via the OpenAI API as of mid-2024.',
        'quality':      'bad',   # pricing info not in KB
    },
    {
        'question':     'How do you implement multi-hop reasoning in a RAG pipeline?',
        'ground_truth': 'Multi-hop RAG decomposes complex questions into sub-questions, retrieves '
                        'evidence for each sub-question separately, and chains the results across '
                        'multiple retrieval steps before generating a final answer. Common approaches '
                        'include iterative retrieval and graph-based reasoning over documents.',
        'quality':      'bad',   # completely absent from KB
    },
]

print(f'Evaluation set: {len(eval_set)} samples')
for i, s in enumerate(eval_set):
    print(f"  Q{i+1} [{s['quality']:6s}] {s['question'][:65]}")

## 7. Run RAG on All Questions

In [ ]:
print('Running RAG pipeline...\n')
for i, sample in enumerate(eval_set):
    answer, contexts = rag(sample['question'], n_results=3)
    sample['answer']   = answer
    sample['contexts'] = contexts
    print(f"Q{i+1} [{sample['quality']:6s}]: {sample['question'][:60]}")
    print(f"Answer        : {answer[:120]}")
    print()

print('RAG complete for all samples.')

## 8. Build RAGAS EvaluationDataset

RAGAS v0.2+ requires `SingleTurnSample` objects wrapped in an `EvaluationDataset` — not a raw HuggingFace `Dataset`.

In [ ]:
ragas_samples = [
    SingleTurnSample(
        user_input         = s['question'],
        response           = s['answer'],
        retrieved_contexts = s['contexts'],   # list[str]
        reference          = s['ground_truth'],
    )
    for s in eval_set
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)
print(f'EvaluationDataset: {len(ragas_dataset.samples)} samples')

---
## 9. Evaluate — Metric by Metric

---
### 9.1 Faithfulness

Checks every claim in the answer against the retrieved context.  
Expected pattern: `good` > `medium` > `bad`  
When the model says "The context does not contain this information", it makes no claims, so faithfulness may be `NaN` or `1.0` — we handle that below.

#### Why Faithfulness = 1.0 for ALL samples (including "bad" ones) — THIS IS CORRECT

Faithfulness measures: **"Can every claim in the answer be traced back to the retrieved context?"**

- **Good/Medium questions:** The model's answer is derived directly from the retrieved chunks → every claim IS in the context → score = 1.0. This is correct.
- **Bad questions:** The model responds with *"The context does not contain this information."* This sentence makes **zero factual claims** about the topic. RAGAS decomposes the answer into atomic claims and finds none to verify → 0 claims, 0 unsupported claims → faithfulness = 1.0 (vacuously true). This is also correct.

**In other words:** Faithfulness = 1.0 everywhere means the RAG pipeline has **zero hallucination** — the model never fabricated information beyond what the context provided. That's the ideal behavior. If you want to see faithfulness < 1.0, you'd need to remove the strict system prompt ("Answer using ONLY the context") so the model starts adding claims not in the context.

**The metrics that DO vary and reveal quality differences are:** `answer_relevancy`, `context_recall`, `context_precision`, and `answer_correctness`.

In [ ]:
res_faith = evaluate(dataset=ragas_dataset, metrics=[metric_faithfulness])
df_faith  = res_faith.to_pandas()

print('=' * 60)
print('FAITHFULNESS')
print('=' * 60)
for i, row in df_faith.iterrows():
    q = eval_set[i]
    score = row.get('faithfulness', float('nan'))
    print(f"  Q{i+1} [{q['quality']:6s}]  {score:.4f}  |  {q['question'][:55]}")

print(f"\n  Mean Faithfulness = {df_faith['faithfulness'].mean():.4f}")

---
### 9.2 Answer Relevancy

Measures how well the answer addresses the question using embedding similarity.  
"The context does not contain this information" responses score **low** here.

In [ ]:
res_rel = evaluate(dataset=ragas_dataset, metrics=[metric_answer_relevancy])
df_rel  = res_rel.to_pandas()

print('=' * 60)
print('ANSWER RELEVANCY')
print('=' * 60)
for i, row in df_rel.iterrows():
    q = eval_set[i]
    score = row.get('answer_relevancy', float('nan'))
    print(f"  Q{i+1} [{q['quality']:6s}]  {score:.4f}  |  {q['question'][:55]}")

print(f"\n  Mean Answer Relevancy = {df_rel['answer_relevancy'].mean():.4f}")

---
### 9.3 Context Precision

Are the retrieved chunks actually relevant to the ground truth?  
For `bad` questions the retriever pulls tangentially-related chunks → **low precision**.

In [ ]:
res_prec = evaluate(dataset=ragas_dataset, metrics=[metric_context_precision])
df_prec  = res_prec.to_pandas()

print('=' * 60)
print('CONTEXT PRECISION')
print('=' * 60)
for i, row in df_prec.iterrows():
    q = eval_set[i]
    score = row.get('llm_context_precision_with_reference', float('nan'))
    print(f"  Q{i+1} [{q['quality']:6s}]  {score:.4f}  |  {q['question'][:55]}")

print(f"\n  Mean Context Precision = {df_prec['llm_context_precision_with_reference'].mean():.4f}")

---
### 9.4 Context Recall

Does the retrieved context cover all the facts in the ground truth?  
When the answer is absent from the KB, recall is **near 0**.

In [ ]:
res_rec = evaluate(dataset=ragas_dataset, metrics=[metric_context_recall])
df_rec  = res_rec.to_pandas()

print('=' * 60)
print('CONTEXT RECALL')
print('=' * 60)
for i, row in df_rec.iterrows():
    q = eval_set[i]
    score = row.get('context_recall', float('nan'))
    print(f"  Q{i+1} [{q['quality']:6s}]  {score:.4f}  |  {q['question'][:55]}")

print(f"\n  Mean Context Recall = {df_rec['context_recall'].mean():.4f}")

### **Answer Correctness**

In [ ]:
# No need to re-import or re-instantiate — boeing clients and metrics are already set up above

res_corr = evaluate(dataset=ragas_dataset, metrics=[metric_answer_correctness])
df_corr  = res_corr.to_pandas()

print('=' * 60)
print('ANSWER CORRECTNESS')
print('=' * 60)
for i, row in df_corr.iterrows():
    q = eval_set[i]
    score = row.get('answer_correctness', float('nan'))
    print(f"  Q{i+1} [{q['quality']:6s}]  {score:.4f}  |  {q['question'][:55]}")

print(f"\n  Mean Answer Correctness = {df_corr['answer_correctness'].mean():.4f}")

---
## 10. Full Combined Evaluation

In [ ]:
all_metrics = [
    metric_faithfulness,
    metric_answer_relevancy,
    metric_context_precision,
    metric_context_recall,
    metric_answer_correctness,
]

res_all = evaluate(dataset=ragas_dataset, metrics=all_metrics)
df_all  = res_all.to_pandas()

# ── Debug: see exactly what columns RAGAS returned ──────────────────────────
print("Actual columns:", df_all.columns.tolist())

# ── Auto-detect metric columns (exclude non-metric cols) ────────────────────
non_metric_cols = {'user_input', 'retrieved_contexts', 'response', 'reference',
                   'question', 'answer', 'contexts', 'ground_truth', 'ground_truths'}
metric_cols = [c for c in df_all.columns if c not in non_metric_cols]
print("Detected metric cols:", metric_cols)

# ── Attach quality tier ──────────────────────────────────────────────────────
df_all['quality']        = [s['quality']               for s in eval_set]
df_all['question_short'] = [s['question'][:48] + '...' for s in eval_set]

display_cols = ['quality', 'question_short'] + metric_cols
print('\nFULL RAGAS RESULTS')
print('=' * 110)
print(df_all[display_cols].to_string(index=False))

print('\nMEAN SCORES BY QUALITY TIER')
print(df_all.groupby('quality')[metric_cols].mean().round(4).to_string())

print('\nOVERALL MEAN SCORES')
for col in metric_cols:
    val = df_all[col].mean()
    bar = '|' * int(val * 25)
    print(f'  {col:<35}: {val:.4f}  {bar}')

## 11. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('RAGAS Evaluation  |  ChromaDB + gpt-4o-mini', fontsize=14, fontweight='bold')

metric_colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2']
tier_color    = {'good': '#2ecc71', 'medium': '#f39c12', 'bad': '#e74c3c'}

# --- Plot 1: overall mean per metric ---
means = df_all[metric_cols].mean()
bars  = axes[0].bar(range(len(metric_cols)), means.values,
                    color=metric_colors, edgecolor='white', linewidth=1.4, width=0.6)
axes[0].set_xticks(range(len(metric_cols)))
axes[0].set_xticklabels([m.replace('_', '\n') for m in metric_cols], fontsize=8)
axes[0].set_ylim(0, 1.18)
axes[0].set_ylabel('Score (0-1)')
axes[0].set_title('Overall Mean per Metric', fontweight='bold')
axes[0].axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
for bar, val in zip(bars, means.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# --- Plot 2: heatmap ---
mat = df_all[metric_cols].values
im  = axes[1].imshow(mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(metric_cols)))
axes[1].set_xticklabels([m.replace('_', '\n') for m in metric_cols], fontsize=7)
ylabels = [f"Q{i+1} [{s['quality']}]" for i, s in enumerate(eval_set)]
axes[1].set_yticks(range(len(eval_set)))
axes[1].set_yticklabels(ylabels, fontsize=8)
axes[1].set_title('Per-Question Heatmap', fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Score')
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        v = mat[r, c]
        axes[1].text(c, r, f'{v:.2f}', ha='center', va='center',
                     fontsize=8, fontweight='bold',
                     color='black' if 0.25 < v < 0.8 else 'white')

# --- Plot 3: grouped bars by quality tier ---
tier_means = df_all.groupby('quality')[metric_cols].mean()
tiers = [t for t in ['good', 'medium', 'bad'] if t in tier_means.index]
x     = np.arange(len(metric_cols))
w     = 0.25
for idx, tier in enumerate(tiers):
    vals = tier_means.loc[tier].values
    axes[2].bar(x + idx*w, vals, w, label=tier.capitalize(),
                color=tier_color[tier], alpha=0.85, edgecolor='white')
axes[2].set_xticks(x + w)
axes[2].set_xticklabels([m.replace('_', '\n') for m in metric_cols], fontsize=7)
axes[2].set_ylim(0, 1.2)
axes[2].set_ylabel('Mean Score')
axes[2].set_title('Mean Score by Quality Tier', fontweight='bold')
axes[2].legend(fontsize=8)
axes[2].axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig('ragas_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as ragas_results.png')

## 12. Export to CSV

In [ ]:
# Build a clean export dataframe
export_rows = []
for i, s in enumerate(eval_set):
    row = {
        'quality':       s['quality'],
        'question':      s['question'],
        'answer':        s['answer'],
        'ground_truth':  s['ground_truth'],
    }
    for col in metric_cols:
        row[col] = df_all[col].iloc[i] if col in df_all.columns else None
    export_rows.append(row)

df_export = pd.DataFrame(export_rows)
df_export.to_csv('ragas_results.csv', index=False)
print('Saved to ragas_results.csv')
df_export

---
## Summary

| Quality tier | Why scores vary |
|---|---|
| **good** | Answer fully in KB → high faithfulness, recall, precision, correctness |
| **medium** | KB has partial info → model fills gaps, may hallucinate → mixed scores |
| **bad** | Answer not in KB → model says "I don't know" → low relevancy & correctness; context recall near 0 |

**Why Faithfulness = 1.0 for all:** The strict system prompt ("Answer using ONLY the context") ensures the model never fabricates claims. For "bad" questions it refuses to answer, making zero claims → vacuously faithful. This is correct behavior, not a bug.

**Key RAGAS v0.2+ changes vs v0.1:**
- Metrics are **class instances** (`Faithfulness()`, not `faithfulness`)
- Dataset is `EvaluationDataset(samples=[SingleTurnSample(...), ...])` — not a HuggingFace `Dataset`
- Field names: `user_input`, `response`, `retrieved_contexts`, `reference` (not `question`, `answer`, `contexts`, `ground_truths`)
- ChromaDB: use `EphemeralClient()` not deprecated `Client()`
- `LLMContextPrecisionWithReference` replaces the old `context_precision` when you have ground truth labels

**Boeing AI Changes:**
- Replaced `ChatOpenAI` → `BoeingChatModel` (with `udal_pat` from `.env`)
- Replaced `OpenAIEmbeddings` → `BoeingEmbeddings`
- Replaced `OpenAIEmbeddingFunction` → custom `BoeingEmbeddingFunction` wrapper for ChromaDB
- Replaced `OpenAI()` client → `BoeingChatModel` with `.invoke()` for RAG generation
- No OpenAI API key needed — uses `UDAL_PAT` from `.env`